# [Chapter 8 — Parameter Estimation, §8.2] The Infected-Viewpoint Estimator $\hat\alpha = J/I$

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 8 — Parameter Estimation
**Considerations developed:** 4 (force of vs from), 8 (parameter estimation)
**Estimated runtime:** ≤ 30 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Demonstrates the infected-viewpoint estimator on synthetic data and verifies it recovers the true transmission rate per infectious person without needing assumptions about the susceptible population.

## What you should already know
Chapter 8 inverse-problem notebook.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import (
    book_style,
    BOOK_COLORS,
    integrate_sir_i,
    baseline_central_comparison,
)
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance

set_seed_chapter_08()
book_style()


## The estimator

If we observe both incidence $J(t)$ and infectious population $I(t)$, the infected-viewpoint estimator is:

$$\hat\alpha(t) = \frac{J(t)}{I(t)}$$

This is the per-infectious-person rate of producing new infections. From Chapter 2B, the true value is $\alpha(t) = c_I \beta (S(t)/N)$, which equals the **effective transmission rate** at the current susceptible level.

The key property: $\hat\alpha$ requires **no assumption** about $S^*$.

In [ ]:
params = baseline_chapter_08()
true_R0 = params['c_I'] * params['beta'] * params['tau_R']
result = integrate_sir_i(params)
t, S, I = result['t'], result['S'], result['I']

# True alpha(t) — what the formula should recover
alpha_true = params['c_I'] * params['beta'] * (S / params['N'])
J_true = alpha_true * I

# Noiseless test: alpha_hat = J / I should recover alpha_true exactly
alpha_hat_noiseless = J_true / np.maximum(I, 1e-12)
mask = (t > 5) & (t < 20)
alpha_hat_noiseless_avg = alpha_hat_noiseless[mask].mean()
alpha_true_avg = alpha_true[mask].mean()

print(f"Noiseless verification (alpha_hat = J/I should equal alpha):")
print(f"  alpha_true (avg over t in (5,20)): {alpha_true_avg:.6f}")
print(f"  alpha_hat = J/I:                   {alpha_hat_noiseless_avg:.6f}")
assert_within_tolerance(alpha_hat_noiseless_avg, alpha_true_avg, rel_tol=1e-6)
print("✅ Formula verified on noiseless data")
print()

# With noise, demonstrate the estimator
J_obs = np.maximum(J_true + params['noise_sigma'] * J_true.max() * np.random.randn(len(t)), 0)
alpha_hat = J_obs / np.maximum(I, 1e-10)

# Use ratio-of-means estimator over a window where I is substantial (less noise-sensitive)
mask = (t > 15) & (t < 35)
alpha_hat_avg = J_obs[mask].mean() / I[mask].mean()
alpha_true_in_window = alpha_true[mask].mean()

print(f"Noisy estimate (ratio of means, t in (15,35)):")
print(f"  alpha_true (avg over window):  {alpha_true_in_window:.5f}")
print(f"  alpha_hat (J_obs.mean / I.mean): {alpha_hat_avg:.5f}")
print(f"  Relative error: {abs(alpha_hat_avg - alpha_true_in_window)/alpha_true_in_window*100:.2f}%")

# Note: alpha = c_I * beta * (S/N), so alpha varies in time as S decreases.
# To recover R_0 = c_I * beta * tau_R, we need alpha / (S/N):
S_avg_window = S[mask].mean() / params['N']
R0_estimate = alpha_hat_avg * params['tau_R'] / S_avg_window
print(f"\nR_0 estimate: alpha_hat * tau_R / (S/N) = {R0_estimate:.3f}")
print(f"True R_0:                                 {true_R0:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(t, alpha_true, color=BOOK_COLORS['alpha_hat'], lw=2, label=r'$\alpha(t)$ true')
ax.plot(t, alpha_hat, '.', color=BOOK_COLORS['neutral'], alpha=0.3, ms=3, label=r'$\hat\alpha = J/I$')
ax.set_xlabel('Time (days)')
ax.set_ylabel(r'$\alpha$ (per day)')
ax.set_title(r'Infected-viewpoint estimator $\hat\alpha = J/I$')
ax.legend()
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()


## What's next

`03_lambda_estimator_baseline.ipynb` does the parallel exercise for the susceptible-viewpoint estimator $\hat\lambda$, and then `04_central_comparison` runs them head-to-head under uncertainty in $S^*$.